# baseline

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV

In [ ]:
# 1. 데이터 로드

df = pd.read_csv("../../data/processed/ljh_preprocessed.csv")

In [ ]:
# 전처리

cols_to_drop = ['user_id', 'reg_date', 'first_deposit', 'first_bet', 'country_id']
df_clean = df.drop(columns=cols_to_drop)

In [ ]:
# 결측치 처리
# 기본 정보(성별, 연령대)가 없는 행 제거
df_clean = df_clean.dropna(subset=['gender', 'age_group'])
 
#활동 정보(hit_days, win_rate 등)가 없는 경우 0으로 채움 (활동 없음을 의미)
df_clean = df_clean.fillna(0)

In [ ]:
# 특성(X)과 타겟(y) 분리
X = df_clean.drop('churn', axis=1)
y = df_clean['churn']

In [ ]:
y.value_counts()

In [ ]:
# 학습/테스트 데이터 분리 (8:2)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
# 스케일링 (로지스틱 회귀는 스케일링 필수)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# 모델 학습 및 평가 

dummy_model = DummyClassifier(strategy='most_frequent')
dummy_model.fit(X_train, y_train)
dummy_score = dummy_model.score(X_test, y_test)

print(f"Baseline Model Accuracy: {dummy_score:.4f}")

In [ ]:
# 2. Logistic Regression Model (로지스틱 회귀)
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_scaled, y_train)

In [ ]:
# 예측
y_pred = lr_model.predict(X_test_scaled)

In [ ]:
# 예측 확률 (ROC-AUC)
y_pred_proba = lr_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
# 평가 결과 출력
accuracy = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, y_pred_proba)  

print(f"Logistic Regression Accuracy: {accuracy:.4f}")
print(f"ROC-AUC Score: {roc_auc:.4f}")  # 
print("-" * 50)
print("Classification Report:")
print(classification_report(y_test, y_pred))

# 하이퍼 파라미터 튜닝  
- 그리드 서치 

In [ ]:
# 1. 튜닝할 파라미터 설정
# 'C': 규제 강도 (값이 작을수록 규제가 강함 = 모델이 단순해짐, 값이 클수록 규제가 약함 = 모델이 복잡해짐)
param_grid = {
    'C': [0.001, 0.01, 0.1, 1, 10, 100],
    'solver': ['lbfgs'], 
    'max_iter': [1000]   
}

In [ ]:
# 2. GridSearchCV 설정
# cv=5: 데이터를 5개로 쪼개서 교차 검증 (과적합 방지)
# scoring='roc_auc': 평가 기준을 ROC-AUC 점수로 설정 (불균형 데이터에 유리)
grid_search = GridSearchCV(LogisticRegression(random_state=42), 
                           param_grid, 
                           cv=5, 
                           #scoring='roc_auc', 
                           #scoring='recall',
                           scoring='f1',
                           n_jobs=-1) 

In [ ]:
# 3. 학습 수행 (최적의 파라미터 찾기)

grid_search.fit(X_train_scaled, y_train)

In [ ]:
# 결과 확인

print(f"최적의 파라미터 (Best Params): {grid_search.best_params_}")
print(f"최고 ROC-AUC 점수 (Best CV Score): {grid_search.best_score_:.4f}")

In [ ]:
# 최적의 모델 저장
best_model = grid_search.best_estimator_

# 테스트 데이터로 예측
y_pred_best = best_model.predict(X_test_scaled)
y_pred_proba_best = best_model.predict_proba(X_test_scaled)[:, 1]

In [ ]:
print("Option: Tuned Logistic Regression Performance")
print(f"Accuracy: {accuracy_score(y_test, y_pred_best):.4f}")
print(f"ROC-AUC: {roc_auc_score(y_test, y_pred_proba_best):.4f}")
print("Classification Report:")
print(classification_report(y_test, y_pred_best))

# 특성 중요도

In [ ]:
# 1. 특성 중요도(회귀 계수) 추출
# lr_model.coef_[0] : 각 특성에 곱해지는 가중치(계수)
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Coefficient': lr_model.coef_[0]
})

In [ ]:
# 2. 절대값 기준으로 정렬하거나, 값의 크기 순으로 정렬
# 해석을 위해 값의 크기 순(양수/음수)으로 정렬합니다.
feature_importance = feature_importance.sort_values(by='Coefficient', ascending=True)

In [ ]:
# 3. 상위 중요 특성 출력
print("--- 이탈 확률을 높이는 주요 요인 (Top 5 Positive) ---")
print(feature_importance.sort_values(by='Coefficient', ascending=False).head(5))

print("\n--- 이탈 확률을 낮추는 주요 요인 (Top 5 Negative) ---")
print(feature_importance.head(5))